# Handwriting Generator — Colab Training

CS 6384 — IAM dataset · ScrabbleGAN + Diffusion

This notebook trains both models on Google Colab with **checkpoint resume** so that Colab disconnects don't lose progress. On every cell re-run, training continues from the most recent checkpoint stored in Drive.

**Before you start, on your laptop:**
1. Push the `handwriting-generator` repo to GitHub (public or private with a token).
2. Upload the IAM dataset (the `data/iam/` directory) to Google Drive at the path you'll set in the CONFIG cell below.
3. Set Runtime → Change runtime type → **GPU** (T4 free, A100 on Pro).

**On reconnect after a session timeout:** just open the notebook and run all cells again. Auto-resume picks up from the last checkpoint.

## 1. Setup

In [ ]:
# Confirm GPU is attached. If this errors, runtime type isn't set to GPU.
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Edit the values below to match your setup. The defaults assume:
- Your repo is at `https://github.com/<you>/handwriting-generator.git`
- IAM dataset uploaded to `MyDrive/handwriting-generator/data/iam/`
- Checkpoints will be written to `MyDrive/handwriting-generator/checkpoints/`

In [ ]:
import os

# === EDIT THESE ===
GIT_REPO_URL = 'https://github.com/YOUR_USERNAME/handwriting-generator.git'
GIT_BRANCH   = 'working-pipeline'

DRIVE_ROOT      = '/content/drive/MyDrive/handwriting-generator'
DRIVE_DATA_DIR  = f'{DRIVE_ROOT}/data/iam'           # where you uploaded IAM
DRIVE_CKPT_DIR  = f'{DRIVE_ROOT}/checkpoints'        # checkpoints persist here

# Training hyperparameters (override smoke defaults for real training)
DIFF_EPOCHS = 100
DIFF_BATCH  = 16          # T4 should handle 16; A100 can do 32
GAN_EPOCHS  = 100
GAN_BATCH   = 32          # GAN is lighter; bump higher on A100
EVAL_SAMPLES = 500

# Local working paths (don't usually need to change)
REPO_DIR  = '/content/handwriting-generator'
LOCAL_DATA_DIR = '/content/data/iam'                 # fast local copy of dataset

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Config OK.')
print(f'  Repo:       {GIT_REPO_URL} (branch {GIT_BRANCH})')
print(f'  Drive data: {DRIVE_DATA_DIR}')
print(f'  Drive ckpt: {DRIVE_CKPT_DIR}')

In [ ]:
# Clone repo (idempotent — re-running just pulls latest)
import os, subprocess
if not os.path.exists(REPO_DIR):
    !git clone -b $GIT_BRANCH $GIT_REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

%cd $REPO_DIR

In [ ]:
!pip install -q -r requirements.txt

## 3. Stage the dataset

Reading IAM images directly from Drive is **very slow** (per-file network round-trip × 115k files). We copy the dataset to local Colab disk once per session — subsequent reads are fast SSD.

First run takes ~3–10 min depending on dataset size and Drive throughput. Re-runs in the same session are instant (skip via cache check).

In [ ]:
import os, shutil, time

if os.path.exists(LOCAL_DATA_DIR) and os.path.exists(f'{LOCAL_DATA_DIR}/splits/train.txt'):
    print(f'Dataset already staged at {LOCAL_DATA_DIR}; skipping copy.')
else:
    print(f'Copying {DRIVE_DATA_DIR} → {LOCAL_DATA_DIR} …')
    t0 = time.time()
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    !cp -r $DRIVE_DATA_DIR/. $LOCAL_DATA_DIR/
    print(f'  Done in {time.time()-t0:.1f}s')

!ls $LOCAL_DATA_DIR
!ls $LOCAL_DATA_DIR/splits 2>/dev/null || echo 'WARNING: no splits/ dir — run prepare_splits.py first'

In [ ]:
# If splits don't exist yet, generate them (one-time per dataset)
import os
if not os.path.exists(f'{LOCAL_DATA_DIR}/splits/train.txt'):
    !python prepare_splits.py --data_root $LOCAL_DATA_DIR
    # Copy the splits back to Drive so they persist
    !mkdir -p $DRIVE_DATA_DIR/splits
    !cp $LOCAL_DATA_DIR/splits/*.txt $DRIVE_DATA_DIR/splits/
else:
    print('Splits already exist; skipping.')

## 4. Train diffusion model

Writes checkpoints directly to Drive so they survive disconnects. `--resume_from auto` picks up from the latest `epoch_*.pth` on every re-run.

**Wall time estimate:** A100 ~8–14 hrs · T4 ~24–36 hrs (across multiple sessions).

In [ ]:
DIFF_OUT = f'{DRIVE_CKPT_DIR}/diffusion'

!python train_diffusion.py \
    --data_root $LOCAL_DATA_DIR \
    --samples_dir data/samples \
    --output_dir $DIFF_OUT \
    --epochs $DIFF_EPOCHS \
    --batch_size $DIFF_BATCH \
    --resume_from auto

## 5. Train GAN

**Wall time estimate:** A100 ~3–5 hrs · T4 ~10–14 hrs.

In [ ]:
GAN_OUT = f'{DRIVE_CKPT_DIR}/gan'

!python train_gan.py \
    --data_root $LOCAL_DATA_DIR \
    --samples_dir data/samples \
    --output_dir $GAN_OUT \
    --epochs $GAN_EPOCHS \
    --batch_size $GAN_BATCH \
    --resume_from auto

## 6. Evaluation

Computes FID, CER, WER, SSIM on the test split. Results write to Drive as `metrics.json`.

In [ ]:
DIFF_RESULTS = f'{DRIVE_ROOT}/results/diffusion'
GAN_RESULTS  = f'{DRIVE_ROOT}/results/gan'

!python evaluate.py \
    --model diffusion \
    --checkpoint $DIFF_OUT/best.pth \
    --data_root $LOCAL_DATA_DIR \
    --style_dir data/samples \
    --output_dir $DIFF_RESULTS \
    --n_samples $EVAL_SAMPLES \
    --save_images

!python evaluate.py \
    --model gan \
    --checkpoint $GAN_OUT/best.pth \
    --data_root $LOCAL_DATA_DIR \
    --style_dir data/samples \
    --output_dir $GAN_RESULTS \
    --n_samples $EVAL_SAMPLES \
    --save_images

In [ ]:
# Show the metrics side-by-side
import json

with open(f'{DIFF_RESULTS}/metrics.json') as f:
    diff_m = json.load(f)
with open(f'{GAN_RESULTS}/metrics.json') as f:
    gan_m = json.load(f)

print(f'{"Metric":<8}{"Diffusion":>12}{"GAN":>12}')
for k in ('FID', 'CER', 'WER', 'SSIM'):
    print(f'{k:<8}{diff_m[k]:>12.4f}{gan_m[k]:>12.4f}')

## 7. Sync TEST_LOG.md back to Drive

If the smoke / training scripts updated `TEST_LOG.md`, persist it to Drive.

In [ ]:
import os, shutil
if os.path.exists('TEST_LOG.md'):
    shutil.copy('TEST_LOG.md', f'{DRIVE_ROOT}/TEST_LOG.md')
    print(f'Copied TEST_LOG.md → {DRIVE_ROOT}/TEST_LOG.md')
    !tail -40 TEST_LOG.md

## 8. Download checkpoints to your laptop

Optional — you can also just open Drive in a browser and download. This cell zips both `best.pth` files into one archive and triggers a browser download.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/best_checkpoints', 'zip',
                    root_dir=DRIVE_CKPT_DIR,
                    base_dir='.')
files.download('/content/best_checkpoints.zip')